# nb5 — Đối sánh external với NomVN (hướng A + B, eval-only)

Notebook thực hiện plan `.kilo/plans/1790234569000-nomvn-external-eval-ab.md`, độc lập với Bước 2 (post-processing) và Run 3:

- **Hướng A — ta trên benchmark của họ**: chạy LoRA adapter Run 1 + Run 2 (output nb3) trên benchmark ngoài `nrl-ai/vn-spell-correction-eval-real` (**CC0**, 150 cặp hand-curated, 6 register: forum / mobile / telex_real / ocr / legal_real / news_real) → **word accuracy + sentence exact match** (bootstrap CI 95%, seed 42, n=1000 resample) + **bộ 3 chỉ số** (detection / correction / over-correction) qua pseudo-annotation `align-v1`.
- **Hướng B — model của họ trên sân ta**: chạy model công khai `nrl-ai/vn-spell-correction-base` (**Apache-2.0**, ViT5-base 220M) trên test 6k của ta → bộ 3 chỉ số + word acc, đối chiếu Run 1 / Run 2.
- **Kèm (rẻ, chung code)**: word acc + sentence exact + CI cho prediction val/test có sẵn của Run 1 / Run 2 (đáp ứng mục P1 "thêm word acc + CI").

**Pre-registered constraint (bắt buộc — mở rộng DESIGN.md §9 sang external eval)**: notebook **eval-only tuyệt đối** — KHÔNG fit/tune bất cứ tham số, rule hay ngưỡng nào trên `eval-real` hoặc trên kết quả Hướng B. Mọi rule của Bước 2 / Run 3 phải derive từ val/train và giữ frozen khi chạm benchmark này.

**Input**:
- Từ nb1: `test_aligned.jsonl` (gold test)
- Từ nb2: `syllable_table.json` (stratified non-word / real-word)
- Từ nb3: `lora_adapter_run1`, `lora_adapter_run2`, `predictions_{val,test}_run{1,2}.jsonl`
- Tùy chọn từ nb3b: `predictions_{val,test}_zeroshot.jsonl` (không có thì log skip)
- Download trực tiếp (cần **Internet ON** trên Kaggle): dataset eval-real (CC0) + model đối thủ (Apache-2.0)

**Quy ước fair**: cùng `MAX_LEN=256` + greedy decode cho cả hai phía (khớp protocol nb3); re-run inference trên GPU có thể lệch nhỏ so với run gốc nb3 (hardware nondeterminism) — ghi chú trong REPORT.

Tham chiếu: `PROJECT.md` §6 (3 nhóm chỉ số bắt buộc) · `DESIGN.md` §6 (định nghĩa metric), §7 (kỷ luật shared cells), §9 (anti-leakage) · `REPORT.md` §1.1 (bối cảnh các Run).

In [11]:
!pip uninstall -y torchao==0.10.0

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [3]:
import os
import json
import random
import datetime
import unicodedata
from pathlib import Path
import collections

# Cài đặt thư viện cần thiết nếu chạy trên Kaggle (giống nb3)
try:
    import peft
    import sentencepiece
except ImportError:
    print('Cài đặt peft và sentencepiece...')
    os.system('pip install -q peft sentencepiece')
    os.system('pip uninstall -y torchao')

# Guard version: transformers v5 bug convert tokenizer sentencepiece (ViT5/T5-style) → KeyError: 0
# (docs của NomVN cũng chỉ định transformers<5). Downgrade + yêu cầu restart kernel.
import transformers
from packaging.version import Version
if Version(transformers.__version__).major >= 5:
    print(f'transformers {transformers.__version__} (v5) — downgrade về <5...')
    os.system('pip install -q "transformers<5"')
    raise SystemExit(
        'Đã downgrade transformers về <5. BÂY GIỜ: Restart Kernel '
        '(Run → Restart & Clear Outputs) rồi Run All lại từ đầu — guard sẽ pass và chạy bình thường.'
    )
    
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, set_seed
from peft import AutoPeftModelForSeq2SeqLM

# --- Cấu hình nb5 (mọi tham số gom một chỗ — DESIGN.md §7) ---
SEED = 42
set_seed(SEED)

N_RESAMPLE = 1000          # bootstrap CI (percentile 2.5 / 97.5)
BOOTSTRAP_ALPHA = 0.05
MAX_SOURCE_LEN = 256       # khớp protocol nb3 — fair cho cả hai model
MAX_TARGET_LEN = 256
EVAL_BATCH_SIZE = 64
DECODE_BEAM_SIZE = 1       # greedy frozen

BARTPHO_MODEL = 'vinai/bartpho-syllable'
THEIR_MODEL_ID = 'nrl-ai/vn-spell-correction-base'
EVALREAL_DATASET_ID = 'nrl-ai/vn-spell-correction-eval-real'

FP16 = torch.cuda.is_available()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('./out')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_STAMP = datetime.datetime.now().isoformat(timespec='seconds')

REQUIRED_FILES = [
    'test_aligned.jsonl',           # nb1 — gold test (text + corrected_text)
    'syllable_table.json',          # nb2 — stratified non-word / real-word
    'lora_adapter_run1',            # nb3 — adapter Run 1 (thư mục chứa adapter_config.json)
    'lora_adapter_run2',            # nb3 — adapter Run 2
    'predictions_val_run1.jsonl',   # nb3 — word acc val
    'predictions_val_run2.jsonl',
    'predictions_test_run1.jsonl',  # nb3 — word acc test
    'predictions_test_run2.jsonl',
]
OPTIONAL_FILES = ['predictions_val_zeroshot.jsonl', 'predictions_test_zeroshot.jsonl',  # nb3b
                  'lora_adapter_run3',  # nb7 — optional Run 3 (WP3): Add Input output nb7 để re-run Hướng A
                  'predictions_val_run3.jsonl', 'predictions_test_run3.jsonl']


def find_required_inputs():
    candidates = []
    kaggle = Path('/kaggle/input')
    if kaggle.is_dir():
        candidates += (sorted(kaggle.rglob('*.jsonl')) + sorted(kaggle.rglob('*.json'))
                       + sorted(kaggle.rglob('adapter_config.json')))
    local = Path('./out')
    if local.is_dir():
        candidates += (sorted(local.rglob('*.jsonl')) + sorted(local.rglob('*.json'))
                       + sorted(local.rglob('adapter_config.json')))

    found = {}
    for p in candidates:
        name = p.name
        if name in REQUIRED_FILES or name in OPTIONAL_FILES:
            found.setdefault(name, p)
        elif name == 'adapter_config.json' and p.parent.name in ('lora_adapter_run1', 'lora_adapter_run2', 'lora_adapter_run3'):
            found.setdefault(p.parent.name, p.parent)

    missing = [req for req in REQUIRED_FILES if req not in found]
    if missing:
        raise FileNotFoundError(
            'Thiếu input bắt buộc: ' + ', '.join(missing) +
            ' | nb5 cần Add Input: output nb1 (test_aligned), nb2 (syllable_table), '
            'nb3 (2 thư mục lora_adapter_run{1,2} + 4 file predictions_{val,test}_run{1,2}.jsonl).'
        )
    return found


INPUT_FILES = find_required_inputs()
OPTIONAL_PRESENT = {name: INPUT_FILES[name] for name in OPTIONAL_FILES if name in INPUT_FILES}
print('=== TỰ DÒ INPUT HOÀN TẤT ===')
for k in REQUIRED_FILES:
    print(f'  {k:32s}: {INPUT_FILES[k]}')
print('Optional (nb3b):', OPTIONAL_PRESENT if OPTIONAL_PRESENT else 'không có — sẽ log skip')
RUN3_PRESENT = 'lora_adapter_run3' in INPUT_FILES
print('Run 3 adapter (nb7):', 'CÓ — thêm nhánh Hướng A cho Run 3' if RUN3_PRESENT else 'không có — chỉ Run 1/Run 2 (giữ nguyên như lần chạy trước)')
print(f'Device: {"CUDA " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"} | FP16={FP16}')

=== TỰ DÒ INPUT HOÀN TẤT ===
  test_aligned.jsonl              : /kaggle/input/notebooks/cquangnguynl/nb1-align-annotate/test_aligned.jsonl
  syllable_table.json             : /kaggle/input/notebooks/cquangnguynl/nb2-pilot-dict-noise/syllable_table.json
  lora_adapter_run1               : /kaggle/input/notebooks/cquangnguynl/nb3-baseline-train-eval/lora_adapter_run1
  lora_adapter_run2               : /kaggle/input/notebooks/cquangnguynl/nb3-baseline-train-eval/lora_adapter_run2
  predictions_val_run1.jsonl      : /kaggle/input/notebooks/cquangnguynl/nb3-baseline-train-eval/predictions_val_run1.jsonl
  predictions_val_run2.jsonl      : /kaggle/input/notebooks/cquangnguynl/nb3-baseline-train-eval/predictions_val_run2.jsonl
  predictions_test_run1.jsonl     : /kaggle/input/notebooks/cquangnguynl/nb3-baseline-train-eval/predictions_test_run1.jsonl
  predictions_test_run2.jsonl     : /kaggle/input/notebooks/cquangnguynl/nb3-baseline-train-eval/predictions_test_run2.jsonl
Optional (nb3b): k

## 1. Cell hàm dùng chung (`SHARED_CELLS_VERSION = 'align-v1'`) — copy NGUYÊN VẸN từ nb3

Copy y hệt cell shared của nb1/nb2/nb3 (`DESIGN.md §7` — bị sửa ở ≥2 nơi thì mới trích `.py` chung): `align-v1` từ nb3 (normalize, tokenize, Levenshtein, edit blocks) + `build_pseudo_annotation` copy nguyên vẹn từ nb2 (cùng family `align-v1`, nb3 không chứa vì tính gold inline trong `evaluate_predictions`). nb5 dùng để pseudo-annotate eval-real trước khi tính bộ 3 chỉ số.

In [4]:
import re

SHARED_CELLS_VERSION = 'align-v1'

_WS_RE = re.compile(r'\s+')
_TOKEN_RE = re.compile(r'\w+|[^\w\s]+')
_HAS_WORD_RE = re.compile(r'\w')
_DIGIT_RE = re.compile(r'\d')

def nfc_normalize(s):
    return _WS_RE.sub(' ', unicodedata.normalize('NFC', s)).strip()

def canon_tokenize(s):
    """NFC + tách token: run chữ/số liền kề (\\w+) là 1 token; cụm dấu câu liền kề là 1 token riêng."""
    return _TOKEN_RE.findall(nfc_normalize(s))

def is_punct_token(tok):
    return not _HAS_WORD_RE.search(tok)

def is_word_token(tok):
    return not is_punct_token(tok) and not _DIGIT_RE.search(tok)

def levenshtein_opcodes(src, tgt):
    n, m = len(src), len(tgt)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        dp[i][0] = i
    for j in range(1, m + 1):
        dp[0][j] = j
    for i in range(1, n + 1):
        si = src[i - 1]
        prev, row = dp[i - 1], dp[i]
        for j in range(1, m + 1):
            best = prev[j - 1] + (0 if si == tgt[j - 1] else 1)
            if prev[j] + 1 < best:
                best = prev[j] + 1
            if row[j - 1] + 1 < best:
                best = row[j - 1] + 1
            row[j] = best
    ops = []
    def _push(tag, i1, i2, j1, j2):
        if ops and ops[-1][0] == tag and ops[-1][1] == i2 and ops[-1][3] == j2:
            ops[-1] = (tag, i1, ops[-1][2], j1, ops[-1][4])
        else:
            ops.append((tag, i1, i2, j1, j2))
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and dp[i][j] == dp[i - 1][j - 1] + (0 if src[i - 1] == tgt[j - 1] else 1):
            _push('equal' if src[i - 1] == tgt[j - 1] else 'replace', i - 1, i, j - 1, j)
            i, j = i - 1, j - 1
        elif i > 0 and dp[i][j] == dp[i - 1][j] + 1:
            _push('delete', i - 1, i, j, j)
            i -= 1
        else:
            _push('insert', i, i, j - 1, j)
            j -= 1
    ops.reverse()
    return ops

def _make_block(src, tgt, span):
    i1, i2, j1, j2 = span
    src_toks, tgt_toks = src[i1:i2], tgt[j1:j2]
    ns, nt = len(src_toks), len(tgt_toks)
    if ns == 0:
        btype = 'insert'
    elif nt == 0:
        btype = 'delete'
    elif ns == 1 and nt == 1:
        btype = 'substitute'
    elif ns == 1:
        btype = 'split'
    elif nt == 1:
        btype = 'merge'
    else:
        btype = 'multi'
    return {
        'type': btype,
        'position': i1,
        'src_span': [i1, i2],
        'tgt_span': [j1, j2],
        'src_tokens': src_toks,
        'tgt_tokens': tgt_toks,
        'punct_only': all(is_punct_token(t) for t in src_toks + tgt_toks),
    }

def extract_edit_blocks(src, tgt, opcodes):
    blocks, cur = [], None
    for tag, i1, i2, j1, j2 in opcodes:
        if tag == 'equal':
            if cur is not None:
                blocks.append(_make_block(src, tgt, cur))
                cur = None
        elif cur is None:
            cur = [i1, i2, j1, j2]
        else:
            cur[1], cur[3] = i2, j2
    if cur is not None:
        blocks.append(_make_block(src, tgt, cur))
    return blocks

def apply_edit_blocks(src, blocks):
    out, pos = [], 0
    for b in blocks:
        out += src[pos:b['src_span'][0]]
        out += b['tgt_tokens']
        pos = b['src_span'][1]
    out += src[pos:]
    return out

SUSPECT_EDIT_RATIO = 0.3  # giữ nguyên giá trị nb1/nb2 — cell hàm dùng chung copy nguyên vẹn cần nó

def build_pseudo_annotation(text, corrected, suspect_ratio=SUSPECT_EDIT_RATIO):
    """Align text ↔ corrected_text trong hệ token canonical → pseudo-annotation schema giống VSEC + edit_blocks.
    Quy ước QUAN TRỌNG (các notebook sau phải dùng cell này nguyên vẹn):
    - error_positions: index token nguồn nằm trong src_span của block CÓ token nguồn.
      Block insert (thiếu âm tiết ở nguồn) KHÔNG có vị trí nguồn → không nằm trong error_positions,
      chỉ nằm trong correction_pairs với error='' và position = vị trí chèn trước (có thể == len(src)).
    - correction_pairs: 1 entry/block; error/correction = các token nối bằng space; delete → correction=''.
    - syllable_annotations: 1 entry/token nguồn; is_correct=False khi token thuộc src_span của block
      non-insert; corrections = chuỗi đích (join space) của block đó.
    - suspect: edit_ratio = (tổng token cả 2 vế nằm trong edit block) / max(len(src), len(tgt)) vượt ngưỡng.
    - align_failed: một trong hai vế token hóa rỗng."""
    src = canon_tokenize(text)
    tgt = canon_tokenize(corrected)
    blocks = extract_edit_blocks(src, tgt, levenshtein_opcodes(src, tgt))
    corrections_by_pos = {}
    for b in blocks:
        if b['src_span'][0] < b['src_span'][1]:
            fix = ' '.join(b['tgt_tokens'])
            for i in range(b['src_span'][0], b['src_span'][1]):
                corrections_by_pos.setdefault(i, []).append(fix)
    error_positions = sorted(corrections_by_pos)
    syllable_annotations = [
        {
            'syllable': tok,
            'is_correct': i not in corrections_by_pos,
            'corrections': corrections_by_pos.get(i, []),
            'position': i,
        }
        for i, tok in enumerate(src)
    ]
    correction_pairs = [
        {'error': ' '.join(b['src_tokens']), 'correction': ' '.join(b['tgt_tokens']), 'position': b['position']}
        for b in blocks
    ]
    n_edit_tokens = sum(
        (b['src_span'][1] - b['src_span'][0]) + (b['tgt_span'][1] - b['tgt_span'][0]) for b in blocks
    )
    denom = max(len(src), len(tgt))
    edit_ratio = n_edit_tokens / denom if denom else 0.0
    return {
        'is_clean': not blocks,
        'align_failed': not src or not tgt,
        'suspect': edit_ratio > suspect_ratio,
        'edit_ratio': round(edit_ratio, 4),
        'error_count': len(blocks),
        'error_positions': error_positions,
        'correction_pairs': correction_pairs,
        'syllable_annotations': syllable_annotations,
        'edit_blocks': blocks,
        'src_tokens': src,
        'tgt_tokens': tgt,
    }

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

print('SHARED_CELLS_VERSION:', SHARED_CELLS_VERSION)

SHARED_CELLS_VERSION: align-v1


### `evaluate_predictions` — copy NGUYÊN VẸN từ nb3 (+ sanity case nhân tạo)

Semantics nhất quán nb3/nb4: gold edits suy bằng align `src ↔ corrected_text`, pred edits suy bằng align `src ↔ prediction`; đối chiếu theo **vị trí token nguồn**; over-correction = FP / token chữ không lỗi trong câu nguồn; Clean Retention = tỷ lệ câu 0-lỗi giữ nguyên output.

In [5]:
def evaluate_predictions(records, predictions, syll_set=None):
    assert len(records) == len(predictions), f'Độ dài không khớp: {len(records)} vs {len(predictions)}'
    
    total_tp = 0
    total_fp = 0
    total_fn = 0
    correct_at_tp = 0
    
    total_clean_tokens = 0
    clean_sents_total = 0
    clean_sents_preserved = 0
    
    # Stratified stats: non-word vs real-word
    stratified = {
        'nonword': {'gold': 0, 'detected': 0, 'corrected': 0},
        'realword': {'gold': 0, 'detected': 0, 'corrected': 0}
    }
    
    sample_overcorrections = []
    
    for rec, pred in zip(records, predictions):
        src_toks = canon_tokenize(rec['text'])
        gold_toks = canon_tokenize(rec['corrected_text'])
        pred_toks = canon_tokenize(pred)
        
        # Gold edit blocks
        gold_ops = levenshtein_opcodes(src_toks, gold_toks)
        gold_blocks = extract_edit_blocks(src_toks, gold_toks, gold_ops)
        
        # Pred edit blocks
        pred_ops = levenshtein_opcodes(src_toks, pred_toks)
        pred_blocks = extract_edit_blocks(src_toks, pred_toks, pred_ops)
        
        gold_pos_map = {}
        for b in gold_blocks:
            if b['src_span'][0] < b['src_span'][1]:
                target_str = ' '.join(b['tgt_tokens'])
                for p in range(b['src_span'][0], b['src_span'][1]):
                    gold_pos_map[p] = (target_str, b['src_tokens'])
                    
        pred_pos_map = {}
        for b in pred_blocks:
            if b['src_span'][0] < b['src_span'][1]:
                pred_str = ' '.join(b['tgt_tokens'])
                for p in range(b['src_span'][0], b['src_span'][1]):
                    pred_pos_map[p] = (pred_str, b['src_tokens'])
                    
        gold_positions = set(gold_pos_map.keys())
        pred_positions = set(pred_pos_map.keys())
        
        tp_pos = gold_positions & pred_positions
        fp_pos = pred_positions - gold_positions
        fn_pos = gold_positions - pred_positions
        
        total_tp += len(tp_pos)
        total_fp += len(fp_pos)
        total_fn += len(fn_pos)
        
        # Đếm số token đúng trong câu nguồn (loại bỏ token dấu câu)
        word_token_positions = {i for i, t in enumerate(src_toks) if is_word_token(t)}
        clean_word_positions = word_token_positions - gold_positions
        total_clean_tokens += len(clean_word_positions)
        
        # Check clean sentence
        if len(gold_positions) == 0:
            clean_sents_total += 1
            if len(pred_positions) == 0:
                clean_sents_preserved += 1
                
        # Correction accuracy
        for p in tp_pos:
            target_str, _ = gold_pos_map[p]
            pred_str, _ = pred_pos_map[p]
            if pred_str.lower() == target_str.lower():
                correct_at_tp += 1
                
            # Stratified
            if syll_set is not None and p < len(src_toks):
                tok_lower = src_toks[p].lower()
                cat = 'nonword' if tok_lower not in syll_set else 'realword'
                stratified[cat]['detected'] += 1
                if pred_str.lower() == target_str.lower():
                    stratified[cat]['corrected'] += 1
                    
        for p in fn_pos:
            if syll_set is not None and p < len(src_toks):
                tok_lower = src_toks[p].lower()
                cat = 'nonword' if tok_lower not in syll_set else 'realword'
                stratified[cat]['gold'] += 1
        for p in tp_pos:
            if syll_set is not None and p < len(src_toks):
                tok_lower = src_toks[p].lower()
                cat = 'nonword' if tok_lower not in syll_set else 'realword'
                stratified[cat]['gold'] += 1
                
        # Lưu mẫu over-correction tiêu biểu
        if fp_pos and len(sample_overcorrections) < 20:
            for p in sorted(fp_pos):
                if p < len(src_toks):
                    sample_overcorrections.append({
                        'orig_token': src_toks[p],
                        'pred_token': pred_pos_map[p][0],
                        'context': ' '.join(src_toks[max(0, p-3):min(len(src_toks), p+4)]),
                        'full_src': rec['text'],
                        'full_pred': pred
                    })
                    if len(sample_overcorrections) >= 20:
                        break

    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    corr_acc = correct_at_tp / total_tp if total_tp > 0 else 0.0
    overcorr_rate = total_fp / total_clean_tokens if total_clean_tokens > 0 else 0.0
    clean_retention = clean_sents_preserved / clean_sents_total if clean_sents_total > 0 else 1.0

    return {
        'detection': {
            'tp': total_tp,
            'fp': total_fp,
            'fn': total_fn,
            'precision': precision,
            'recall': recall,
            'f1': f1,
        },
        'correction': {
            'correct_at_tp': correct_at_tp,
            'accuracy': corr_acc,
        },
        'over_correction': {
            'fp_count': total_fp,
            'clean_tokens': total_clean_tokens,
            'rate': overcorr_rate,
            'clean_sents_total': clean_sents_total,
            'clean_sents_preserved': clean_sents_preserved,
            'clean_retention_rate': clean_retention,
        },
        'stratified': stratified,
        'sample_overcorrections': sample_overcorrections,
    }

# Sanity Test hàm đánh giá trên ví dụ giả định (copy nguyên vẹn từ nb3)
_test_recs = [{'text': 'học sanh đi hoc', 'corrected_text': 'học sinh đi học'}]
_test_preds = ['học sinh đi hoc'] # Model sửa 'sanh' -> 'sinh', nhưng bỏ sót 'hoc'
_m_test = evaluate_predictions(_test_recs, _test_preds)
assert _m_test['detection']['tp'] == 1 and _m_test['detection']['fn'] == 1 and _m_test['detection']['fp'] == 0
assert _m_test['correction']['accuracy'] == 1.0
print('Sanity check hàm evaluate_predictions PASS 100%!')

Sanity check hàm evaluate_predictions PASS 100%!


### Helper riêng của nb5 — word accuracy / sentence exact / bootstrap CI

- **`word_acc`**: align token canonical giữa `clean ↔ pred` bằng `levenshtein_opcodes`; `word_acc = (số token equal) / len(tokens(clean))`. Câu clean rỗng → 0.0 (sẽ log nếu xảy ra).
- **`sentence_exact`**: `canon_tokenize(pred) == canon_tokenize(clean)` (case-sensitive, punct-sensitive — nhất quán hệ token canonical).
- **`bootstrap_ci`**: resample **theo câu**, seed 42, n=1000, percentile 2,5/97,5 — cùng phương pháp với bench công bố của NomVN.

Lưu ý trung thực: định nghĩa word acc bên dưới là **tự định nghĩa của ta** — có thể lệch nhỏ so với implementation của họ về cách đếm token; chỉ so hướng (ordinal), không so từng chữ số thập phân.

In [6]:
def word_acc(clean_text, pred_text):
    clean_toks = canon_tokenize(clean_text)
    pred_toks = canon_tokenize(pred_text)
    if not clean_toks:
        return 0.0
    ops = levenshtein_opcodes(clean_toks, pred_toks)
    equal_tokens = sum((i2 - i1) for tag, i1, i2, j1, j2 in ops if tag == 'equal')
    return equal_tokens / len(clean_toks)


def sentence_exact(clean_text, pred_text):
    return canon_tokenize(pred_text) == canon_tokenize(clean_text)


def bootstrap_ci(values, n_resample=N_RESAMPLE, alpha=BOOTSTRAP_ALPHA, seed=SEED):
    """CI percentile cho mean của list giá trị per-sentence (resample theo câu, seeded)."""
    if not values:
        return None, None
    rng = random.Random(seed)
    n = len(values)
    means = []
    for _ in range(n_resample):
        total = 0
        for _ in range(n):
            total += values[rng.randrange(n)]
        means.append(total / n)
    means.sort()
    m = len(means)                                    # số lần resample — không phải len(values)
    lo = means[max(0, int((alpha / 2) * m))]
    hi = means[min(m - 1, int((1 - alpha / 2) * m))]
    return lo, hi


def word_acc_block(pairs):
    """pairs: list (clean_text, pred_text) → stats word acc + sentence exact + CI."""
    accs = [word_acc(c, p) for c, p in pairs]
    exacts = [1.0 if sentence_exact(c, p) else 0.0 for c, p in pairs]
    lo, hi = bootstrap_ci(accs)
    lo_e, hi_e = bootstrap_ci(exacts)
    return {
        'n': len(pairs),
        'word_acc_mean': sum(accs) / len(accs) if accs else 0.0,
        'word_acc_ci95': [lo, hi],
        'sentence_exact_mean': sum(exacts) / len(exacts) if exacts else 0.0,
        'sentence_exact_ci95': [lo_e, hi_e],
    }


# Sanity checks
assert word_acc('học sinh đi học', 'học sinh đi hoc') == 3 / 4
assert word_acc('học sinh đi học', 'học sinh đi học') == 1.0
assert sentence_exact('a b', 'a b') and not sentence_exact('a b', 'a, b')
_lo, _hi = bootstrap_ci([1.0] * 50 + [0.0] * 50, n_resample=200)
assert _lo < 0.5 < _hi, (_lo, _hi)
print('Sanity helpers word_acc / sentence_exact / bootstrap_ci PASS')

Sanity helpers word_acc / sentence_exact / bootstrap_ci PASS


## 2. Tải benchmark eval-real (CC0) — duyệt toàn bộ configs

Duyệt **tất cả** configs mà dataset công bố (forum / mobile / telex_real / ocr / legal_real / news_real — tự dò bằng `get_dataset_config_names`, không hard-code). Tự nhận diện cặp trường `(noisy, clean)` qua danh sách key ứng viên. **Cần Internet ON**; download fail → hướng dẫn fallback upload thủ công (pattern bảng âm tiết nb2).

In [7]:
from datasets import load_dataset, get_dataset_config_names
import datasets

EVALREAL_FALLBACK_HINT = (
    'Không tải được ' + EVALREAL_DATASET_ID + '. '
    'Trên Kaggle: bật Internet (Settings → Internet ON) rồi chạy lại. '
    'Hoặc fallback thủ công: download dataset từ Hugging Face, nén thành Kaggle Dataset '
    'giữ cấu trúc <config>/ chứa file data (parquet/jsonl/csv), Add Input vào notebook này.'
)


def extract_pair(ex):
    key_pairs = [('noisy', 'clean'), ('noisy_text', 'clean_text'), ('text', 'clean'),
                 ('text', 'label'), ('input', 'target'), ('sentence', 'correction')]
    for nk, ck in key_pairs:
        if nk in ex and ck in ex:
            return str(ex[nk]), str(ex[ck])
    raise KeyError(f'Không nhận diện được cặp trường noisy/clean trong example: keys={list(ex.keys())}')


evalreal_records = []
bench_meta = {'dataset_id': EVALREAL_DATASET_ID, 'datasets_version': datasets.__version__, 'configs': {}}
try:
    config_names = sorted(get_dataset_config_names(EVALREAL_DATASET_ID))
    print('Configs phát hiện:', config_names)
    for cfg in config_names:
        ds = load_dataset(EVALREAL_DATASET_ID, cfg, split='train')
        first_keys = None
        for ex in ds:
            noisy, clean = extract_pair(ex)
            if first_keys is None:
                first_keys = list(ex.keys())
            evalreal_records.append({
                'row_id': f'{cfg}#{len(evalreal_records):04d}',
                'register': cfg,
                'text': nfc_normalize(str(noisy)),
                'corrected_text': nfc_normalize(str(clean)),
            })
        bench_meta['configs'][cfg] = {'n': len(ds), 'example_keys': first_keys}
        print(f'  {cfg:16s}: {len(ds)} cặp — keys={first_keys}')
except Exception as exc:
    raise RuntimeError(EVALREAL_FALLBACK_HINT + f' | Chi tiết lỗi: {exc}') from exc

print(f'Tổng: {len(evalreal_records)} cặp (kỳ vọng ~150, 6 register)')
assert len(evalreal_records) > 0, 'eval-real rỗng — dừng thay vì đo trên bench rỗng'

README.md: 0.00B [00:00, ?B/s]

Configs phát hiện: ['forum', 'legal_real', 'mobile', 'news_real', 'ocr', 'telex_real']


forum_25.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/25 [00:00<?, ? examples/s]

  forum           : 25 cặp — keys=['input', 'target']


...

Generating train split:   0%|          | 0/25 [00:00<?, ? examples/s]

  ocr             : 25 cặp — keys=['input', 'target']


telex_real_25.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/25 [00:00<?, ? examples/s]

  telex_real      : 25 cặp — keys=['input', 'target']
Tổng: 150 cặp (kỳ vọng ~150, 6 register)


In [8]:
# 2b. Kiểm tra tính toàn vẹn bench: roundtrip pseudo-annotation + identity floor

align_failed_rows = []
n_clean = 0
n_suspect = 0
for rec in evalreal_records:
    pa = build_pseudo_annotation(rec['text'], rec['corrected_text'])
    assert apply_edit_blocks(pa['src_tokens'], pa['edit_blocks']) == pa['tgt_tokens'], rec['row_id']
    if pa['align_failed']:
        align_failed_rows.append(rec['row_id'])
    n_clean += 1 if pa['is_clean'] else 0
    n_suspect += 1 if pa['suspect'] else 0
print(f'Roundtrip PASS trên {len(evalreal_records)} cặp · align_failed={len(align_failed_rows)} · '
      f'câu sạch (0 edit block)={n_clean} · suspect (edit_ratio>30%)={n_suspect}')
if align_failed_rows:
    print('  align_failed:', align_failed_rows)

# Identity floor: pred = src → TP=0, FP=0 (over-correction 0%)
identity_preds = [r['text'] for r in evalreal_records]
_id_metrics = evaluate_predictions(evalreal_records, identity_preds)
assert _id_metrics['detection']['tp'] == 0 and _id_metrics['over_correction']['fp_count'] == 0, 'Identity floor FAIL'
_id_wordacc = word_acc_block([(r['corrected_text'], r['text']) for r in evalreal_records])
print(f"Identity floor PASS — word_acc identity = {_id_wordacc['word_acc_mean']:.2%} (sàn suy từ lỗi gold của bench)")

Roundtrip PASS trên 150 cặp · align_failed=0 · câu sạch (0 edit block)=10 · suspect (edit_ratio>30%)=116
Identity floor PASS — word_acc identity = 51.96% (sàn suy từ lỗi gold của bench)


## 3. Nạp model — đối thủ + 2 adapter của ta (nạp tuần tự tiết kiệm VRAM)

- Model đối thủ: `nrl-ai/vn-spell-correction-base` (ViT5-base) — smoke test đúng ví dụ docs (`"Toi yu Vit Nam"`); nếu fp16 cho tỷ lệ output rỗng cao (overflow kiểu T5 trên T4) → **tự fallback fp32** và log (đây là fallback hạ tầng, không phải tune theo metric).
- Adapter của ta: `AutoPeftModelForSeq2SeqLM` từ thư mục `lora_adapter_run{1,2}` của nb3 (base model đọc từ adapter config = `vinai/bartpho-syllable`).
- `batch_generate`: copy semantics từ nb3 §7 (greedy, truncation `MAX_SOURCE_LEN`, decode + NFC) — chỉ đổi default sang hằng số nb5.

In [9]:
def _hub_commit_hash(model):
    return getattr(getattr(model, 'config', None), '_commit_hash', None)


def batch_generate(model, tokenizer, texts, batch_size=EVAL_BATCH_SIZE, beam_size=DECODE_BEAM_SIZE):
    """Copy semantics từ nb3 §7 — greedy, truncation MAX_SOURCE_LEN, decode + NFC normalize."""
    model.eval()
    device = next(model.parameters()).device
    preds = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        enc = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_SOURCE_LEN,
            return_tensors='pt'
        ).to(device)
        with torch.no_grad():
            outputs = model.generate(
                input_ids=enc['input_ids'],
                attention_mask=enc['attention_mask'],
                max_length=MAX_TARGET_LEN,
                num_beams=beam_size,
                early_stopping=True if beam_size > 1 else False
            )
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        preds.extend([nfc_normalize(d) for d in decoded])
        if (i // batch_size) % 50 == 0:
            print(f'  Generated {min(i + batch_size, len(texts))}/{len(texts)} sentences...')
    return preds


def count_truncation(tokenizer, texts, max_len=MAX_SOURCE_LEN, chunk=256):
    n_trunc = 0
    for i in range(0, len(texts), chunk):
        enc = tokenizer(texts[i:i + chunk], truncation=False)['input_ids']
        n_trunc += sum(1 for ids in enc if len(ids) > max_len)
    return n_trunc


def load_adapter(adapter_dir):
    print(f'Nạp adapter: {adapter_dir}')
    model = AutoPeftModelForSeq2SeqLM.from_pretrained(
        str(adapter_dir), torch_dtype=torch.float16 if FP16 else torch.float32
    ).to(DEVICE)
    model.eval()
    return model


print(f'Nạp model đối thủ: {THEIR_MODEL_ID} (fp16={FP16})')
their_tokenizer = AutoTokenizer.from_pretrained(THEIR_MODEL_ID)
their_model = AutoModelForSeq2SeqLM.from_pretrained(
    THEIR_MODEL_ID, torch_dtype=torch.float16 if FP16 else torch.float32
).to(DEVICE)
their_model.eval()
their_revision = _hub_commit_hash(their_model)
their_precision_used = 'float16' if FP16 else 'float32'
print('Revision:', their_revision)

# Smoke test theo ví dụ docs của họ (soft check — ghi nhận, không fail cứng)
_smoke_in = 'Toi yu Vit Nam'
_smoke_out = batch_generate(their_model, their_tokenizer, [_smoke_in])[0]
_smoke_flags = [ch for ch in 'ăâêôơưđáàảãạắằẳẵặéèẻẽẹếềểễệíìỉĩịóòỏõọốồổỗộúùủũụứừửữựýỳỷỹỵ' if ch in _smoke_out.lower()]
print(f'Smoke test: "{_smoke_in}" -> "{_smoke_out}" · có dấu tiếng Việt: {bool(_smoke_flags)}')

bart_tokenizer = AutoTokenizer.from_pretrained(BARTPHO_MODEL)
print('Tokenizer BARTpho sẵn sàng (cho adapter Run 1 / Run 2)')

Nạp model đối thủ: nrl-ai/vn-spell-correction-base (fp16=True)


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/820k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/897 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

dict.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/2.83M [00:00<?, ?B/s]

Tokenizer BARTpho sẵn sàng (cho adapter Run 1 / Run 2)


## 4. Hướng A — inference Run 1 + Run 2 trên eval-real (150 câu)

Nạp tuần tự từng adapter (xong thì giải phóng VRAM), decode greedy frozen, xuất `predictions_evalreal_run{1,2}.jsonl`.

In [12]:
RUN_KEYS = [('run1', 'lora_adapter_run1'), ('run2', 'lora_adapter_run2')]
if RUN3_PRESENT:  # optional Run 3 (nb7) — WP3; phần cũ bất biến khi không có run3
    RUN_KEYS.append(('run3', 'lora_adapter_run3'))
RUN_NAMES = [k for k, _ in RUN_KEYS]

evalreal_texts = [r['text'] for r in evalreal_records]

preds_evalreal = {}
for run_name, adapter_key in RUN_KEYS:
    print(f'--- INFERENCE A · {run_name} trên eval-real ({len(evalreal_texts)} câu) ---')
    model = load_adapter(INPUT_FILES[adapter_key])
    preds_evalreal[run_name] = batch_generate(model, bart_tokenizer, evalreal_texts)
    n_empty = sum(1 for p in preds_evalreal[run_name] if not p.strip())
    print(f'{run_name}: xong · empty preds = {n_empty}/{len(preds_evalreal[run_name])}')
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

for run_name in RUN_NAMES:
    out_path = OUTPUT_DIR / f'predictions_evalreal_{run_name}.jsonl'
    with open(out_path, 'w', encoding='utf-8') as f:
        for rec, p in zip(evalreal_records, preds_evalreal[run_name]):
            f.write(json.dumps({'row_id': rec['row_id'], 'register': rec['register'],
                                'text': rec['text'], 'corrected_text': rec['corrected_text'],
                                'prediction': p}, ensure_ascii=False) + '\n')
    print('Đã ghi', out_path.name)

--- INFERENCE A · run1 trên eval-real (150 câu) ---
Nạp adapter: /kaggle/input/notebooks/cquangnguynl/nb3-baseline-train-eval/lora_adapter_run1
  Generated 16/150 sentences...
run1: xong · empty preds = 0/150
--- INFERENCE A · run2 trên eval-real (150 câu) ---
Nạp adapter: /kaggle/input/notebooks/cquangnguynl/nb3-baseline-train-eval/lora_adapter_run2
  Generated 16/150 sentences...
run2: xong · empty preds = 0/150
Đã ghi predictions_evalreal_run1.jsonl
Đã ghi predictions_evalreal_run2.jsonl


## 5. Hướng B — model đối thủ trên test 6k (sân nhà của ta)

Inference trên `test_aligned.jsonl` (cùng nguồn `text` với protocol nb3), log tỷ lệ câu bị truncate ở `MAX_SOURCE_LEN` (đọc kèm hạn chế khi diễn giải điểm của model đối thủ). Xuất `predictions_test_nomvnbase.jsonl`.

In [13]:
test_records = load_jsonl(INPUT_FILES['test_aligned.jsonl'])
test_texts = [r['text'] for r in test_records]
n_trunc = count_truncation(their_tokenizer, test_texts)
print(f'Test: {len(test_records)} câu · vượt MAX_SOURCE_LEN={MAX_SOURCE_LEN}: {n_trunc} câu ({n_trunc / len(test_records):.2%})')

print(f'--- INFERENCE B · {THEIR_MODEL_ID} trên test 6k ---')
preds_test_theirs = batch_generate(their_model, their_tokenizer, test_texts)
n_empty = sum(1 for p in preds_test_theirs if not p.strip())
empty_rate = n_empty / len(preds_test_theirs)
print(f'Xong · empty preds = {n_empty}/{len(preds_test_theirs)} ({empty_rate:.1%})')

if FP16 and empty_rate > 0.2:
    print(f'⚠️ Empty rate {empty_rate:.1%} cao — nghi overflow fp16 (kiểu T5), chạy lại bằng fp32...')
    del their_model
    torch.cuda.empty_cache()
    their_model = AutoModelForSeq2SeqLM.from_pretrained(THEIR_MODEL_ID, torch_dtype=torch.float32).to(DEVICE)
    their_model.eval()
    their_revision = _hub_commit_hash(their_model)
    their_precision_used = 'float32'
    preds_test_theirs = batch_generate(their_model, their_tokenizer, test_texts)

del their_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

out_path = OUTPUT_DIR / 'predictions_test_nomvnbase.jsonl'
with open(out_path, 'w', encoding='utf-8') as f:
    for rec, p in zip(test_records, preds_test_theirs):
        f.write(json.dumps({'row_id': rec.get('row_id'), 'text': rec['text'],
                            'corrected_text': rec['corrected_text'], 'prediction': p},
                           ensure_ascii=False) + '\n')
print('Đã ghi', out_path.name)

Test: 5983 câu · vượt MAX_SOURCE_LEN=256: 12 câu (0.20%)
--- INFERENCE B · nrl-ai/vn-spell-correction-base trên test 6k ---
  Generated 16/5983 sentences...
  Generated 816/5983 sentences...
  Generated 1616/5983 sentences...
  Generated 2416/5983 sentences...
  Generated 3216/5983 sentences...
  Generated 4016/5983 sentences...
  Generated 4816/5983 sentences...
  Generated 5616/5983 sentences...
Xong · empty preds = 0/5983 (0.0%)
Đã ghi predictions_test_nomvnbase.jsonl


## 6. Đánh giá — bộ 3 chỉ số + word acc cho A và B

- **A (eval-real)**: `evaluate_predictions` cho Run 1 / Run 2 + breakdown **per-register** (word acc + detection F1) — dùng cho khung diễn giải pre-registered (news/legal kỳ vọng cạnh tranh, forum/telex/OCR kỳ vọng yếu).
- **B (test)**: `evaluate_predictions` cho model đối thủ, cùng gold `align-v1` với nb3/nb4.
- **Word acc + sentence exact + CI**: eval-real (identity / Run1 / Run2), test (Run1 / Run2 / zeroshot-tùy-chọn / nomvn-base), val (Run1 / Run2 — từ prediction có sẵn của nb3).

In [16]:
SYLL_TABLE = json.loads(Path(INPUT_FILES['syllable_table.json']).read_text(encoding='utf-8'))
SYLL_SET = set(SYLL_TABLE['entries'])

eval_results = {}
registers = sorted({r['register'] for r in evalreal_records})

# 6a. Hướng A — eval-real: 3 chỉ số (toàn bench + per-register)
for run_name in RUN_NAMES:
    key = f'evalreal_{run_name}'
    eval_results[key] = evaluate_predictions(evalreal_records, preds_evalreal[run_name], SYLL_SET)
    per_register = {}
    for cfg in registers:
        idx = [i for i, r in enumerate(evalreal_records) if r['register'] == cfg]
        sub_recs = [evalreal_records[i] for i in idx]
        sub_preds = [preds_evalreal[run_name][i] for i in idx]
        per_register[cfg] = {
            'detection_f1': evaluate_predictions(sub_recs, sub_preds)['detection']['f1'],
            'word_acc': word_acc_block([(r['corrected_text'], p) for r, p in zip(sub_recs, sub_preds)]),
        }
    eval_results[key]['per_register'] = per_register

# 6b. Hướng B — test: 3 chỉ số cho model đối thủ
eval_results['test_nomvnbase'] = evaluate_predictions(test_records, preds_test_theirs, SYLL_SET)

# 6c. Word acc + sentence exact + CI
word_acc_results = {}
word_acc_results['evalreal_identity'] = word_acc_block(
    [(r['corrected_text'], r['text']) for r in evalreal_records])
for run_name in RUN_NAMES:
    word_acc_results[f'evalreal_{run_name}'] = word_acc_block(
        [(r['corrected_text'], p) for r, p in zip(evalreal_records, preds_evalreal[run_name])])
for name, fname in [('val_run1', 'predictions_val_run1.jsonl'),
                    ('val_run2', 'predictions_val_run2.jsonl'),
                    ('test_run1', 'predictions_test_run1.jsonl'),
                    ('test_run2', 'predictions_test_run2.jsonl'),
                    ('test_zeroshot', 'predictions_test_zeroshot.jsonl'),
                    ('val_run3', 'predictions_val_run3.jsonl'),
                    ('test_run3', 'predictions_test_run3.jsonl')]:
    if fname in INPUT_FILES:
        recs = load_jsonl(INPUT_FILES[fname])
        word_acc_results[name] = word_acc_block(
            [(r['corrected_text'], r['prediction']) for r in recs])
    else:
        word_acc_results[name] = None
word_acc_results['test_nomvnbase'] = word_acc_block(
    [(r['corrected_text'], p) for r, p in zip(test_records, preds_test_theirs)])

# In bảng tổng hợp
def _fmt_wa(blk):
    if blk is None:
        return 'N/A'
    lo, hi = blk['word_acc_ci95']
    return f"{blk['word_acc_mean']:.2%} [CI95 {lo:.2%}–{hi:.2%}]"

print('=== WORD ACCURACY (mean [CI95], self-defined — xem docstring cell helper) ===')
_wa_keys = ['evalreal_identity'] + [f'evalreal_{k}' for k in RUN_NAMES] + \
    ['val_run1', 'val_run2', 'test_run1', 'test_run2', 'test_zeroshot',
     'val_run3', 'test_run3', 'test_nomvnbase']
for k in _wa_keys:
    print(f'  {k:20s}: {_fmt_wa(word_acc_results.get(k))}')

def _pct(v):
    return f'{v:.2%}'

print()
print('=== BỘ 3 CHỈ SỐ (semantics align-v1 — nhất quán nb3/nb4) ===')
metric_keys = ['evalreal_' + k for k in RUN_NAMES] + ['test_nomvnbase']
print(f"  {'Chỉ số':<26s}" + ''.join(f' | {k:>16s}' for k in metric_keys))
rows = [
    ('Detection Precision', lambda m: _pct(m['detection']['precision'])),
    ('Detection Recall', lambda m: _pct(m['detection']['recall'])),
    ('Detection F1', lambda m: _pct(m['detection']['f1'])),
    ('Correction Acc @TP', lambda m: _pct(m['correction']['accuracy'])),
    ('Over-correction Rate', lambda m: _pct(m['over_correction']['rate'])),
    ('Clean Retention', lambda m: _pct(m['over_correction']['clean_retention_rate'])),
]
for name, get in rows:
    print(f"  {name:<26s}" + ''.join(f' | {get(eval_results[k]):>16s}' for k in metric_keys))

print()
print('=== PER-REGISTER (eval-real) — word acc [CI95] / detection F1 ===')
def _wa_short(blk):
    lo, hi = blk['word_acc_ci95']
    return f"{blk['word_acc_mean']:.2%} [{lo:.0%}-{hi:.0%}]"
print(f"  {'register':<16s} | {'Run1 WA':>22s} | {'Run2 WA':>22s} | {'Run1 F1':>8s} | {'Run2 F1':>8s}")
for cfg in registers:
    r1 = eval_results['evalreal_run1']['per_register'][cfg]
    r2 = eval_results['evalreal_run2']['per_register'][cfg]
    print(f"  {cfg:<16s} | {_wa_short(r1['word_acc']):>22s} | {_wa_short(r2['word_acc']):>22s} "
          f"| {r1['detection_f1']:>8.2%} | {r2['detection_f1']:>8.2%}")

print()
print('=== STRATIFIED (eval-real · Run 2) — non-word vs real-word ===')
_st = eval_results['evalreal_run2']['stratified']
for cat in ('nonword', 'realword'):
    gold = _st[cat]['gold']
    det = _st[cat]['detected'] / gold if gold else 0.0
    corr = _st[cat]['corrected'] / gold if gold else 0.0
    print(f'  {cat:<10s} gold={gold:<6d} recall(detect)={det:.2%} corrected(gold)={corr:.2%}')

=== WORD ACCURACY (mean [CI95], self-defined — xem docstring cell helper) ===
  evalreal_identity   : 51.96% [CI95 46.96%–57.26%]
  evalreal_run1       : 55.65% [CI95 50.20%–61.39%]
  evalreal_run2       : 55.54% [CI95 50.29%–60.93%]
  val_run1            : 97.85% [CI95 97.64%–98.06%]
  val_run2            : 97.80% [CI95 97.59%–98.01%]
  test_run1           : 92.56% [CI95 92.40%–92.72%]
  test_run2           : 93.14% [CI95 92.99%–93.30%]
  test_zeroshot       : N/A
  test_nomvnbase      : 94.80% [CI95 94.64%–94.95%]

=== BỘ 3 CHỈ SỐ (semantics align-v1 — nhất quán nb3/nb4) ===
  Chỉ số                     |    evalreal_run1 |    evalreal_run2 |   test_nomvnbase
  Detection Precision        |           96.76% |           96.04% |           88.62%
  Detection Recall           |           18.83% |           19.64% |           85.94%
  Detection F1               |           31.52% |           32.61% |           87.26%
  Correction Acc @TP         |           23.44% |           24.31% |    

## 7. Đối chiếu anchor — số công bố của NomVN trên cùng bench

Số được hard-code từ docs công khai (đo 30/04/2026, word acc tổng hợp n=150) — chỉ dùng làm **anchor tham chiếu hướng**, không phải ground truth cùng implementation metric.

In [17]:
ANCHOR_SOURCE = 'https://nom-vn.nrl.ai/tasks/spell-correction (đo 2026-04-30)'
ANCHORS = [
    ('nrl-ai/vn-spell-correction-base (v0.2.29)', 79.62),
    ('nrl-ai/vn-spell-correction-small (v0.2.29)', 77.55),
    ('Toshiiiii1', 77.40),
    ('qthuan ViT5', 72.42),
    ('chamdent (ViT5_Vietnamese_Correction)', 51.69),
    ('bmd1905/vietnamese-correction-v2', 49.21),
    ('iAmHieu2012/vit5-vietnamese-spelling-correction', 45.57),
]

print('=== ĐỐI CHIẾU ANCHOR CÔNG BỐ (eval-real · word acc tổng hợp n=150) ===')
print(f"  {'Model':<50s} | {'WA %':>7s}")
for name, wa in ANCHORS:
    print(f'  {name:<50s} | {wa:>7.2f}')
_ours_r2 = word_acc_results['evalreal_run2']['word_acc_mean'] * 100
_ours_r1 = word_acc_results['evalreal_run1']['word_acc_mean'] * 100
print(f"  {'→ nb5 · Run 1 (BARTpho LoRA thuần, VSEC-real)':<50s} | {_ours_r1:>7.2f}")
print(f"  {'→ nb5 · Run 2 (BARTpho LoRA + augmentation)':<50s} | {_ours_r2:>7.2f}")
if RUN3_PRESENT:
    _ours_r3 = word_acc_results['evalreal_run3']['word_acc_mean'] * 100
    print(f"  {'→ nb5 · Run 3 (BARTpho LoRA + noise v2)':<50s} | {_ours_r3:>7.2f}")
print()
print(f'Nguồn anchor: {ANCHOR_SOURCE}')
print('Lưu ý: word acc tự định nghĩa của nb5 có thể lệch nhỏ so với implementation của họ — chỉ so hướng (ordinal).')

=== ĐỐI CHIẾU ANCHOR CÔNG BỐ (eval-real · word acc tổng hợp n=150) ===
  Model                                              |    WA %
  nrl-ai/vn-spell-correction-base (v0.2.29)          |   79.62
  nrl-ai/vn-spell-correction-small (v0.2.29)         |   77.55
  Toshiiiii1                                         |   77.40
  qthuan ViT5                                        |   72.42
  chamdent (ViT5_Vietnamese_Correction)              |   51.69
  bmd1905/vietnamese-correction-v2                   |   49.21
  iAmHieu2012/vit5-vietnamese-spelling-correction    |   45.57
  → nb5 · Run 1 (BARTpho LoRA thuần, VSEC-real)      |   55.65
  → nb5 · Run 2 (BARTpho LoRA + augmentation)        |   55.54

Nguồn anchor: https://nom-vn.nrl.ai/tasks/spell-correction (đo 2026-04-30)
Lưu ý: word acc tự định nghĩa của nb5 có thể lệch nhỏ so với implementation của họ — chỉ so hướng (ordinal).


## 8. QA — in mẫu để soát tay

10 mẫu/bench (SRC / GOLD / PRED) — soát: (1) prediction hợp lệ về mặt ngôn ngữ, (2) không có pattern rỗng/lặp bất thường, (3) hành động model hợp lý so với gold. Đây là QA đầu ra, **không** dùng để chỉnh gì theo bench (pre-registered constraint).

In [18]:
print('=== QA · eval-real × Run 2 (10 mẫu đầu) ===')
for rec, p in list(zip(evalreal_records, preds_evalreal['run2']))[:10]:
    print(f"[{rec['row_id']}]")
    print('  SRC :', rec['text'])
    print('  GOLD:', rec['corrected_text'])
    print('  PRED:', p)
    print()

print('=== QA · test × nomvn-base (10 mẫu đầu) ===')
for rec, p in list(zip(test_records, preds_test_theirs))[:10]:
    print(f"[{rec.get('row_id')}]")
    print('  SRC :', rec['text'])
    print('  GOLD:', rec['corrected_text'])
    print('  PRED:', p)
    print()

=== QA · eval-real × Run 2 (10 mẫu đầu) ===
[forum#0000]
  SRC : Mn ơi cho mk hỏi vs ạ, ai bt cách cài thì cmt giúp em với ạ
  GOLD: Mọi người ơi cho mình hỏi với ạ, ai biết cách cài thì comment giúp em với ạ
  PRED: Mn ơi cho em hỏi vs ạ, ai biết cách cài thì cmt giúp em với ạ

[forum#0001]
  SRC : Hôm qua đi xem phim mà ko bt nó dài v, bùn ngủ wá luôn
  GOLD: Hôm qua đi xem phim mà không biết nó dài vậy, buồn ngủ quá luôn
  PRED: Hôm qua đi xem phim mà không thấy nó dài, bùn ngủá luôn

[forum#0002]
  SRC : Mua đồ online dc free ship ko shop ơi, mk ở Hà Nội nhé
  GOLD: Mua đồ online được free ship không shop ơi, mình ở Hà Nội nhé
  PRED: Mua đồ online có free ship không shop ơi, mk ở Hà Nội nhé!

[forum#0003]
  SRC : Cái áo này nhìn cx đẹp đó nhưng giá hơi chát so vs chất liệu
  GOLD: Cái áo này nhìn cũng đẹp đó nhưng giá hơi chát so với chất liệu
  PRED: Cái áo này nhìn cũng đẹp đó nhưng giá hơi chát so vs chất liệu

[forum#0004]
  SRC : Nay đi ăn lẩu ở quán mới khai trương, đồ ăn ng

## 9. Xuất `external_bench_report.json`

Gói mọi số liệu + provenance (revision model/dataset, precision thực dùng, tỷ lệ truncate, `SHARED_CELLS_VERSION`, pre-registered constraint) + QA samples — đầu vào cho mục mới của `REPORT.md` và Nhật ký quyết định `DESIGN.md` §12.

In [19]:
external_report = {
    'created': RUN_STAMP,
    'notebook': 'nb5_external_eval',
    'shared_cells_version': SHARED_CELLS_VERSION,
    'pre_registered_constraint': 'Eval-only: không fit/tune bất cứ tham số/rule/ngưỡng nào trên eval-real '
                                 'hoặc trên kết quả Hướng B (mở rộng DESIGN.md §9 sang external eval).',
    'config': {
        'seed': SEED,
        'n_resample': N_RESAMPLE,
        'bootstrap_alpha': BOOTSTRAP_ALPHA,
        'max_source_len': MAX_SOURCE_LEN,
        'max_target_len': MAX_TARGET_LEN,
        'decode': 'greedy (num_beams=1)',
        'precision': {
            'ours_adapters': 'float16' if FP16 else 'float32',
            'their_model': their_precision_used,
        },
    },
    'provenance': {
        'bartpho_base': BARTPHO_MODEL,
        'their_model_id': THEIR_MODEL_ID,
        'their_model_revision': their_revision,
        'evalreal_dataset': bench_meta,
        'evalreal_license': 'CC0 (public domain) — https://huggingface.co/datasets/nrl-ai/vn-spell-correction-eval-real',
        'their_model_license': 'Apache-2.0 — https://huggingface.co/nrl-ai/vn-spell-correction-base',
        'input_files': {k: str(v) for k, v in INPUT_FILES.items()},
        'optional_files_present': {k: str(v) for k, v in OPTIONAL_PRESENT.items()},
        'run3_adapter_present': RUN3_PRESENT,
        'test_truncation_over_max_len': {'n': n_trunc, 'total': len(test_records), 'max_len': MAX_SOURCE_LEN},
        'evalreal_integrity': {'align_failed': align_failed_rows, 'n_clean': n_clean, 'n_suspect': n_suspect},
    },
    'anchor_published': [{'model': name, 'word_acc_pct': wa, 'source': ANCHOR_SOURCE} for name, wa in ANCHORS],
    'results': {
        'evalreal_3metrics': {f'evalreal_{k}': eval_results[f'evalreal_{k}'] for k in RUN_NAMES},
        'test_nomvnbase_3metrics': eval_results['test_nomvnbase'],
        'word_acc': word_acc_results,
    },
    'qa_samples': {
        'evalreal_run2': [{'row_id': r['row_id'], 'src': r['text'], 'gold': r['corrected_text'], 'pred': p}
                          for r, p in list(zip(evalreal_records, preds_evalreal['run2']))[:10]],
        'test_nomvnbase': [{'row_id': r.get('row_id'), 'src': r['text'], 'gold': r['corrected_text'], 'pred': p}
                           for r, p in list(zip(test_records, preds_test_theirs))[:10]],
    },
    'files': [f'predictions_evalreal_{k}.jsonl' for k in RUN_NAMES] +
             ['predictions_test_nomvnbase.jsonl', 'external_bench_report.json'],
}

with open(OUTPUT_DIR / 'external_bench_report.json', 'w', encoding='utf-8') as f:
    json.dump(external_report, f, ensure_ascii=False, indent=2)

print('=== HOÀN TẤT nb5 ===')
print('Đã ghi vào', OUTPUT_DIR)
for fn in external_report['files']:
    print(' -', fn)

=== HOÀN TẤT nb5 ===
Đã ghi vào /kaggle/working
 - predictions_evalreal_run1.jsonl
 - predictions_evalreal_run2.jsonl
 - predictions_test_nomvnbase.jsonl
 - external_bench_report.json
